> **Chapter 15, Part 8** | The honesty closer. **Focus:** four ways an orchestrated pipeline reports success while being wrong. Pairs with 12.7, 13.8, and 14.8.

# When the Schedule Lies

Every advanced chapter in this repo ends by naming its own failure modes, because an apparatus that cannot fail cannot inform. Orchestration's failures are worse than cron's in one specific way: a scheduler with a green dashboard is more trusted, so when it lies, it lies with authority.

Four failure modes, each demonstrated.

In [1]:
# The tiny orchestrator we build across 15.1-15.3, collected into one cell.
from collections import deque
from dataclasses import dataclass
from typing import Callable, Optional


@dataclass
class Asset:
    name: str
    deps: list
    compute: Callable
    partitioned: bool = False


class AssetGraph:
    def __init__(self):
        self.assets = {}

    def add(self, asset):
        self.assets[asset.name] = asset
        return self

    def _downstream(self):
        down = {n: [] for n in self.assets}
        for n, a in self.assets.items():
            for d in a.deps:
                if d in down:
                    down[d].append(n)
        return down

    def topological_order(self):
        indeg = {n: 0 for n in self.assets}
        for n, a in self.assets.items():
            for d in a.deps:
                if d in indeg:
                    indeg[n] += 1
        down = self._downstream()
        q = deque(sorted(n for n, k in indeg.items() if k == 0))
        order = []
        while q:
            n = q.popleft()
            order.append(n)
            for m in sorted(down[n]):
                indeg[m] -= 1
                if indeg[m] == 0:
                    q.append(m)
        if len(order) != len(self.assets):
            stuck = sorted(n for n in self.assets if n not in order)
            raise ValueError("cycle detected among assets: " + ", ".join(stuck))
        return order

    def _needed(self, targets):
        need = set()
        stack = list(targets)
        while stack:
            n = stack.pop()
            if n in need or n not in self.assets:   # ignore refs outside the graph
                continue
            need.add(n)
            stack.extend(self.assets[n].deps)
        return need

    def materialize(self, targets=None, log=None, partition=None, verbose=True):
        order = self.topological_order()
        if targets is not None:
            need = self._needed(targets)
            order = [n for n in order if n in need]
        results = {}
        for n in order:
            tag = " [" + str(partition) + "]" if partition is not None else ""
            if log is not None and log.is_materialized(n, partition):
                results[n] = log.value(n, partition)
                if verbose:
                    print("  skip   " + n + tag + " (already materialized)")
                continue
            inputs = {d: results.get(d) for d in self.assets[n].deps}
            value = self.assets[n].compute(inputs)
            results[n] = value
            if log is not None:
                log.record(n, partition, value)
            if verbose:
                print("  build  " + n + tag)
        return results


class MaterializationLog:
    def __init__(self):
        self.store = {}

    def is_materialized(self, name, partition=None):
        return (name, partition) in self.store

    def record(self, name, partition=None, value=None):
        self.store[(name, partition)] = value

    def value(self, name, partition=None):
        return self.store[(name, partition)]


def backfill(graph, target, partitions, log, verbose=True):
    report = {"materialized": [], "skipped": []}
    order = [n for n in graph.topological_order() if n in graph._needed([target])]
    for p in partitions:
        for n in order:
            if log.is_materialized(n, p):
                report["skipped"].append((n, p))
                continue
            inputs = {d: (log.value(d, p) if log.is_materialized(d, p) else None)
                      for d in graph.assets[n].deps}
            log.record(n, p, graph.assets[n].compute(inputs))
            report["materialized"].append((n, p))
    return report


def blast_radius(graph, failed):
    down = graph._downstream()
    seen, stack = set(), list(down[failed])
    while stack:
        n = stack.pop()
        if n in seen:
            continue
        seen.add(n)
        stack.extend(down[n])
    return seen


@dataclass
class RetryPolicy:
    max_attempts: int = 1


def materialize_with_failures(graph, failing=None, retry=None, verbose=True):
    failing = failing or {}
    retry = retry or RetryPolicy()
    order = graph.topological_order()
    status, results, attempts_used = {}, {}, {}
    for n in order:
        if any(status.get(d) in ("failed", "skipped") for d in graph.assets[n].deps):
            status[n] = "skipped"
            if verbose:
                print("  skip    " + n + " (upstream failed)")
            continue
        attempts, ok = 0, False
        while attempts < retry.max_attempts:
            attempts += 1
            if attempts <= failing.get(n, 0):
                if verbose:
                    print("  retry   " + n + " attempt " + str(attempts) + " failed")
                continue
            ok = True
            break
        attempts_used[n] = attempts
        if ok:
            inputs = {d: results.get(d) for d in graph.assets[n].deps}
            results[n] = graph.assets[n].compute(inputs)
            status[n] = "materialized"
            if verbose:
                print("  build   " + n + " (attempt " + str(attempts) + ")")
        else:
            status[n] = "failed"
            if verbose:
                print("  FAIL    " + n + " (exhausted " + str(retry.max_attempts) + " attempts)")
    return status, results, attempts_used


print("orchestrator core ready:", len([Asset, AssetGraph, MaterializationLog,
      backfill, blast_radius, RetryPolicy, materialize_with_failures]), "building blocks")

orchestrator core ready: 7 building blocks


## 1. Silent partial failure

An asset "completes" but produces empty output. The run is green. Downstream builds on nothing and reports zero, which looks like a quiet day rather than a broken pipeline. The fix is a data-quality assertion inside the asset, not a status check outside it.

In [2]:
g = AssetGraph()
g.add(Asset("ingest", [], lambda i: []))                     # returns EMPTY, but does not raise
g.add(Asset("rollup", ["ingest"], lambda i: {"rows": len(i["ingest"])}))

res = g.materialize(verbose=False)
print("run status: SUCCESS (every asset returned without error)")
print("rollup output:", res["rollup"], "<- zero rows, and nothing flagged it")
print()
# The fix: assert inside the asset.
def ingest_checked(i):
    data = []
    if len(data) == 0:
        raise ValueError("ingest produced 0 rows; failing loudly instead of passing empty downstream")
    return data

try:
    AssetGraph().add(Asset("ingest", [], ingest_checked)).materialize(verbose=False)
except ValueError as e:
    print("with an in-asset check:", e)

run status: SUCCESS (every asset returned without error)
rollup output: {'rows': 0} <- zero rows, and nothing flagged it

with an in-asset check: ingest produced 0 rows; failing loudly instead of passing empty downstream


## 2. Sensor storms

A sensor whose condition is always true fires on every poll. Instead of one run per file, you get a run per tick, and the executor drowns. The fix is a cursor: the sensor must remember what it already handled.

In [3]:
class Clock:
    def __init__(self): self.t = 0
    def tick(self): self.t += 1


# Broken: condition ignores whether work was already done.
clock = Clock()
storm = 0
for _ in range(10):
    if True:                      # always fires
        storm += 1
    clock.tick()
print(f"broken sensor: {storm} runs launched in 10 ticks (a storm)")

# Fixed: a cursor remembers the last handled tick.
clock = Clock()
handled_until = -1
fixed_runs = 0
arrivals = {3, 7}
for _ in range(10):
    if clock.t in arrivals and clock.t > handled_until:
        fixed_runs += 1
        handled_until = clock.t
    clock.tick()
print(f"fixed sensor:  {fixed_runs} runs (one per genuine arrival)")

broken sensor: 10 runs launched in 10 ticks (a storm)
fixed sensor:  2 runs (one per genuine arrival)


## 3. Backfill thundering herd

Backfilling a year of daily partitions at once launches 365 simultaneous materializations. The warehouse that handles one day comfortably falls over under 365. The fix is a concurrency limit: backfill in bounded waves.

In [4]:
def backfill_waves(partitions, max_concurrent):
    waves = [partitions[i:i + max_concurrent] for i in range(0, len(partitions), max_concurrent)]
    return waves


year = [f"day_{i:03d}" for i in range(365)]
naive = len(year)
waves = backfill_waves(year, max_concurrent=10)
print(f"naive backfill: {naive} partitions launched at once (thundering herd)")
print(f"bounded backfill: {len(waves)} waves of <= 10, peak concurrency 10")
print("peak warehouse load drops by", f"{naive // 10}x")

naive backfill: 365 partitions launched at once (thundering herd)
bounded backfill: 37 waves of <= 10, peak concurrency 10
peak warehouse load drops by 36x


## 4. Retry masking a data bug

Retries are for transient failures. Point them at a deterministic data bug and they just burn compute, then fail anyway, while delaying the alert by however long the retries took. A retry policy should distinguish retriable errors (timeouts) from terminal ones (a schema violation).

In [5]:
attempts = {"count": 0}


def deterministic_bug(i):
    attempts["count"] += 1
    raise ValueError("column 'price' is null")     # same failure every time, not transient


g = AssetGraph().add(Asset("bad", [], deterministic_bug))
status, _, used = materialize_with_failures(g, failing={"bad": 99},
                                            retry=RetryPolicy(max_attempts=5), verbose=False)
print(f"retried {5} times, still failed; wasted {5 - 1} extra runs on a non-transient bug")
print()
print("the fix: classify errors")
print("  retriable  -> timeout, connection reset, 503    -> retry")
print("  terminal   -> null constraint, schema mismatch  -> fail fast, alert now")

retried 5 times, still failed; wasted 4 extra runs on a non-transient bug

the fix: classify errors
  retriable  -> timeout, connection reset, 503    -> retry
  terminal   -> null constraint, schema mismatch  -> fail fast, alert now


## The through-line

All four failures share a shape: the orchestrator reported success on a metric that was not the thing you cared about. Green status is not green data. The same lesson closed Chapter 12 (descriptors oversold), Chapter 13 (governance signals gamed), and Chapter 14 (benchmark speedups that vanish in production).

An orchestrator is leverage. It runs your pipeline reliably, on a schedule, with recovery. It will also run a wrong pipeline reliably, on a schedule, with recovery. The instrumentation that catches the four failures above (in-asset data checks, sensor cursors, concurrency caps, error classification) is not optional polish. It is the part that makes the leverage safe.

That is Chapter 15. You built an orchestrator, pointed it at the repo's own dbt project and trading platform, mapped it to Dagster, and learned where it bites. Chapter 16 on the roadmap takes the next step: data contracts and change capture, which move quality enforcement to the producer boundary where governance actually has leverage.